In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np

In [15]:
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [16]:
class WatermarkedCNN(nn.Module):
    def __init__(self):
        super(WatermarkedCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=5)  # Embedding target
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(16 * 12 * 12, 10)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x

In [17]:
T = 256
lambda_reg = 0.1  # Strength of watermark regularizer

def generate_watermark(conv_layer, T=256):
    W = conv_layer.weight.data.clone().detach().cpu().numpy()
    M = np.prod(W.shape)  # Flatten all weights
    X = np.random.randn(T, M).astype(np.float32)
    b = np.ones(T, dtype=np.float32)  # Embedding all 1s
    return torch.tensor(X), torch.tensor(b)

In [18]:
def embedding_loss(conv_layer, X, b):
    W_all = conv_layer.weight.view(-1)  # Flatten all weights
    proj = torch.matmul(X, W_all)
    y = torch.sigmoid(proj)
    return nn.BCELoss()(y, b)

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = WatermarkedCNN().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()

In [20]:
X, b = generate_watermark(model.conv1, T)
X, b = X.to(device), b.to(device)

In [21]:
for epoch in range(20):
    model.train()
    total_loss = 0
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss_task = criterion(outputs, labels)
        loss_embed = embedding_loss(model.conv1, X, b)
        loss = loss_task + lambda_reg * loss_embed
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if batch_idx == 0:
            print(f"[Epoch {epoch+1}] Task loss: {loss_task.item():.4f}, Embed loss: {loss_embed.item():.4f}")
    print(f"Epoch {epoch+1}, Total Loss: {total_loss:.4f}")

[Epoch 1] Task loss: 2.3514, Embed loss: 1.2570
Epoch 1, Total Loss: 304.9174
[Epoch 2] Task loss: 0.0872, Embed loss: 0.2217
Epoch 2, Total Loss: 109.6288
[Epoch 3] Task loss: 0.1468, Embed loss: 0.1046
Epoch 3, Total Loss: 77.4460
[Epoch 4] Task loss: 0.0380, Embed loss: 0.0668
Epoch 4, Total Loss: 63.1758
[Epoch 5] Task loss: 0.0788, Embed loss: 0.0477
Epoch 5, Total Loss: 53.6854
[Epoch 6] Task loss: 0.0486, Embed loss: 0.0381
Epoch 6, Total Loss: 47.4003
[Epoch 7] Task loss: 0.0234, Embed loss: 0.0312
Epoch 7, Total Loss: 43.4145
[Epoch 8] Task loss: 0.0101, Embed loss: 0.0262
Epoch 8, Total Loss: 40.0089
[Epoch 9] Task loss: 0.0162, Embed loss: 0.0228
Epoch 9, Total Loss: 36.6453
[Epoch 10] Task loss: 0.0045, Embed loss: 0.0201
Epoch 10, Total Loss: 33.8024
[Epoch 11] Task loss: 0.0523, Embed loss: 0.0176
Epoch 11, Total Loss: 31.6668
[Epoch 12] Task loss: 0.0018, Embed loss: 0.0160
Epoch 12, Total Loss: 29.0158
[Epoch 13] Task loss: 0.0254, Embed loss: 0.0147
Epoch 13, Total Los

In [22]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"\nTest Accuracy: {100 * correct / total:.2f}%")


Test Accuracy: 98.79%


In [23]:
def extract_watermark(conv_layer, X):
    W_all = conv_layer.weight.view(-1)
    proj = torch.matmul(X, W_all)
    return (proj >= 0).float()

with torch.no_grad():
    extracted = extract_watermark(model.conv1, X)
    ber = (extracted != b).float().mean().item()
    print(f"Bit Error Rate (BER): {ber * 100:.2f}%")

Bit Error Rate (BER): 0.00%
